In [18]:
print("This is the begenning of the mini project.")

This is the begenning of the mini project.


In [19]:
import matplotlib.pyplot as plt
import torch
import numpy as np
import pandas as pd
from pathlib import Path

In [25]:
def converge_csvs_to_dataframe(root_folder_path):
    """
    Recursively searches for all CSV files in the given directory and 
    concatenates them into a single pandas DataFrame.
    """
    folder_path = Path(root_folder_path)
    
    # Recursively find all .csv files
    csv_files = list(folder_path.rglob('*.csv'))
    
    if not csv_files:
        print(f"No CSV files found in {root_folder_path}")
        return pd.DataFrame()

    dataframes = []
    i=0
    for file in csv_files:
        if i<10:
            try:
                # Read each CSV
                df = pd.read_csv(file, dtype={'Pleth': 'float32', 'II': 'float32'})

                # Optional: Add a column to track which file the data came from
                df['source_file'] = file.name

                dataframes.append(df)
                print(f"Successfully loaded: {file.name}")
            except Exception as e:
                print(f"Error reading {file.name}: {e}")
            i+=1

    # Converge all dataframes into one
    converged_df = pd.concat(dataframes, ignore_index=True)
    return converged_df

In [26]:
dataset = converge_csvs_to_dataframe(root_folder_path="/home/shaurya/Documents/Mini_Project26/rrest-syn_csv")
print(dataset[:10])

Successfully loaded: rrest-syn076_data.csv
Successfully loaded: rrest-syn085_data.csv
Successfully loaded: rrest-syn082_data.csv
Successfully loaded: rrest-syn051_data.csv
Successfully loaded: rrest-syn148_data.csv
Successfully loaded: rrest-syn044_data.csv
Successfully loaded: rrest-syn121_data.csv
Successfully loaded: rrest-syn180_data.csv
Successfully loaded: rrest-syn176_data.csv
Successfully loaded: rrest-syn135_data.csv
          0  62.893            source_file  -4.412  -757.31  -4.4066  -757.53  \
0  0.000755  62.893  rrest-syn076_data.csv     NaN      NaN      NaN      NaN   
1  0.002483  62.893  rrest-syn076_data.csv     NaN      NaN      NaN      NaN   
2  0.004919  62.893  rrest-syn076_data.csv     NaN      NaN      NaN      NaN   
3  0.007827  62.873  rrest-syn076_data.csv     NaN      NaN      NaN      NaN   
4  0.011298  62.656  rrest-syn076_data.csv     NaN      NaN      NaN      NaN   
5  0.016014  61.778  rrest-syn076_data.csv     NaN      NaN      NaN      NaN   
6  

In [34]:
dataset.shape[0]

1049985

In [31]:
def plot_converged_waveforms(converged_df, fs=500, time_window=None, source_file=None):
    """
    Plots the ECG (Lead II) and PPG (Pleth) waveforms directly from an in-memory 
    converged pandas DataFrame, skipping any file reading.
    """
    # Filter by source file if a specific recording is requested
    if source_file and 'source_file' in converged_df.columns:
        plot_df = converged_df[converged_df['source_file'] == source_file].copy()
    else:
        plot_df = converged_df.copy()

    # Ensure required columns exist
    if 'II' not in plot_df.columns or 'Pleth' not in plot_df.columns:
        raise ValueError("The converged DataFrame must contain 'II' and 'Pleth' columns.")

    # Generate continuous time axis based on the sampling frequency
    num_samples = len(plot_df)
    time_axis = np.arange(num_samples) / fs
    plot_df['Time_sec'] = time_axis

    # Apply time window if specified to prevent overplotting
    if time_window:
        start_sec, end_sec = time_window
        plot_df = plot_df[(plot_df['Time_sec'] >= start_sec) & (plot_df['Time_sec'] <= end_sec)]
        
    # Set up the matplotlib figure
    fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(12, 6), sharex=True)
    
    # Plot Simulated ECG (II)
    axes[0].plot(plot_df['Time_sec'], plot_df['II'], color='red', linewidth=1)
    axes[0].set_title('Simulated ECG (Lead II)', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Amplitude', fontsize=10)
    axes[0].grid(True, linestyle='--', alpha=0.6)
    
    # Plot Simulated Finger PPG (Pleth)
    axes[1].plot(plot_df['Time_sec'], plot_df['Pleth'], color='blue', linewidth=1)
    axes[1].set_title('Simulated Finger PPG (Pleth)', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Amplitude', fontsize=10)
    axes[1].set_xlabel('Time (Seconds)', fontsize=10)
    axes[1].grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

In [32]:
plot_converged_waveforms(dataset)

ValueError: The converged DataFrame must contain 'II' and 'Pleth' columns.